# 🎯 PROJET DATA MINING: Classification de Défaut de Crédit
## Analyse Factorielle Discriminante | Régression Logistique | Arbre de Décision | Random Forest

**Objectif**: Comparer 4 modèles de classification avec analyse statistique complète, sélection de variables, optimisation des hyperparamètres et évaluation rigoureuse.

**Méthodologie**:
1. ✅ Exploration et Nettoyage des Données
2. ✅ Gestion des valeurs manquantes et du déséquilibre
3. ✅ Sélection de Variables (SelectKBest + RFECV)
4. ✅ Modélisation de 4 algorithmes
5. ✅ Fine-tuning des hyperparamètres
6. ✅ Évaluation (ROC/AUC/PR-AUC/Confusion Matrix)
7. ✅ Interprétation (Feature Importance, Odds Ratio, SHAP)

In [ ]:
# 📦 CELL 1: Import des Bibliothèques Nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.metrics import (roc_curve, auc, roc_auc_score, confusion_matrix, 
                             classification_report, accuracy_score, precision_score, 
                             recall_score, f1_score, precision_recall_curve, average_precision_score)
from sklearn.utils import class_weight
from imblearn.over_sampling import SMOTE
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration des styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
print("✅ Toutes les bibliothèques importées avec succès!")

## 📊 PHASE 1: CHARGEMENT ET EXPLORATION DES DONNÉES

In [ ]:
# CELL 2: Chargement du Dataset
df = pd.read_csv('cs-training.csv', index_col=0)
print("📋 INFORMATIONS SUR LE DATASET")
print("="*80)
print(f"Dimensions: {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"\nNoms des variables:\n{df.columns.tolist()}")
print(f"\nVariable cible: 'SeriousDlqin2yrs' (Serious Delinquency in 2 years)")
print(f"\nTypes de données:\n{df.dtypes}")
print(f"\n👀 Aperçu des 5 premières lignes:")
print(df.head())

In [ ]:
# CELL 3: Analyse des Valeurs Manquantes
print("\n❌ ANALYSE DES VALEURS MANQUANTES")
print("="*80)
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Variable': missing_data.index,
    'Manquantes': missing_data.values,
    'Pourcentage': missing_percent.values
})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Manquantes', ascending=False)
print(missing_df)
print(f"\nTotal: {df.isnull().sum().sum()} valeurs manquantes")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
missing_df.set_index('Variable')['Pourcentage'].plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Pourcentage de Valeurs Manquantes par Variable', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pourcentage (%)')

# Déséquilibre des classes
class_dist = df['SeriousDlqin2yrs'].value_counts()
axes[1].bar(class_dist.index, class_dist.values, color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=2)
axes[1].set_title('Distribution de la Variable Cible', fontsize=12, fontweight='bold')
axes[1].set_xlabel('SeriousDlqin2yrs')
axes[1].set_ylabel('Fréquence')
axes[1].set_xticklabels(['Pas de défaut (0)', 'Défaut (1)'])
for i, v in enumerate(class_dist.values):
    axes[1].text(i, v + 1000, f'{v}\n({v/len(df)*100:.2f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📊 DISTRIBUTION DE LA VARIABLE CIBLE:")
print(class_dist)
print(f"\nRatio de déséquilibre: 1:{class_dist[0]/class_dist[1]:.2f}")

In [ ]:
# CELL 4: Statistiques Descriptives Détaillées
print("\n📈 STATISTIQUES DESCRIPTIVES DES VARIABLES")
print("="*80)
print(df.describe().T)

# Détection des outliers avec IQR
def detect_outliers(data):
    outlier_info = {}
    for col in data.select_dtypes(include=[np.number]).columns:
        if col != 'SeriousDlqin2yrs':
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers = data[(data[col] < lower) | (data[col] > upper)][col]
            if len(outliers) > 0:
                outlier_info[col] = {
                    'count': len(outliers),
                    'percent': (len(outliers) / len(data)) * 100,
                    'lower_bound': lower,
                    'upper_bound': upper
                }
    return outlier_info

outliers = detect_outliers(df)
if outliers:
    print("\n⚠️ OUTLIERS DÉTECTÉS (Méthode IQR):")
    for var, info in outliers.items():
        print(f"  {var}: {info['count']} outliers ({info['percent']:.2f}%)")
else:
    print("\n✅ Pas d'outliers détectés significatifs")

## 🧹 PHASE 2: NETTOYAGE ET PRÉPARATION DES DONNÉES

In [ ]:
# CELL 5: Imputation des Valeurs Manquantes
print("\n🔧 IMPUTATION DES VALEURS MANQUANTES")
print("="*80)

# Séparation features et cible
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

# Stratégie d'imputation statistique
print("Avant imputation:")
print(X.isnull().sum()[X.isnull().sum() > 0])

# Pour MonthlyIncome: imputer par la médiane (robuste aux outliers)
X['MonthlyIncome'].fillna(X['MonthlyIncome'].median(), inplace=True)

# Pour NumberOfDependents: imputer par le mode (variable discrète)
X['NumberOfDependents'].fillna(X['NumberOfDependents'].mode()[0], inplace=True)

print("\nAprès imputation:")
print(f"Valeurs manquantes restantes: {X.isnull().sum().sum()}")
print("✅ Imputation complétée")

# Vérification des variables problématiques (DebtRatio)
print(f"\nVérification DebtRatio:")
print(f"  Valeurs négatives: {(X['DebtRatio'] < 0).sum()}")
print(f"  Valeurs extrêmes (>1000): {(X['DebtRatio'] > 1000).sum()}")
# Les valeurs extrêmes peuvent être des codes manquants, on les laisse pour le moment
# On les traitera lors de la normalisation

In [ ]:
# CELL 6: Traitement des Outliers et Winsorization
print("\n⚙️ TRAITEMENT DES VALEURS EXTRÊMES")
print("="*80)

from scipy.stats import mstats

# Winsorization à 1% et 99% pour les variables problématiques
X['DebtRatio'] = mstats.winsorize(X['DebtRatio'], limits=[0.01, 0.01])
X['RevolvingUtilizationOfUnsecuredLines'] = mstats.winsorize(
    X['RevolvingUtilizationOfUnsecuredLines'], limits=[0.01, 0.01]
)

print("✅ Winsorization appliquée aux variables extrêmes")
print("\nNouveaux min/max après Winsorization:")
print(X[['DebtRatio', 'RevolvingUtilizationOfUnsecuredLines']].describe().loc[['min', 'max']])

In [ ]:
# CELL 7: Création de Variables (Feature Engineering)
print("\n🎨 CRÉATION DE NOUVELLES VARIABLES")
print("="*80)

# Création de variables d'interaction et transformations
X['age_squared'] = X['age'] ** 2
X['debt_to_income'] = X['DebtRatio'] / (X['MonthlyIncome'] + 1)  # Éviter division par zéro
X['credit_utilization_risk'] = X['RevolvingUtilizationOfUnsecuredLines'] * X['NumberOfOpenCreditLinesAndLoans']
X['late_payment_score'] = (X['NumberOfTime30-59DaysPastDueNotWorse'] + 
                           X['NumberOfTime60-89DaysPastDueNotWorse'] * 2 + 
                           X['NumberOfTimes90DaysLate'] * 3)

print("Nouvelles variables créées:")
print("  - age_squared: Age au carré")
print("  - debt_to_income: Ratio dette/revenu")
print("  - credit_utilization_risk: Risque d'utilisation du crédit")
print("  - late_payment_score: Score de paiement tardif pondéré")

print(f"\nDimensions finales: {X.shape}")
print(f"Variables: {X.columns.tolist()}")

## 🔍 PHASE 3: SÉLECTION DE VARIABLES

In [ ]:
# CELL 8: Normalisation (Scaling) des Variables
print("\n📏 NORMALISATION DES VARIABLES")
print("="*80)

# Utilisation du RobustScaler pour la robustesse aux outliers
scaler = RobustScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("✅ Variables normalisées avec RobustScaler")
print(f"\nMoyenne et écart-type après normalisation (sur un échantillon):")
print(X_scaled.describe().loc[['mean', 'std']].T.head())

In [ ]:
# CELL 9: Sélection de Variables - SelectKBest (Filtrage)
print("\n🎯 SÉLECTION DE VARIABLES - SELECTKBEST")
print("="*80)

# SelectKBest avec f_classif (ANOVA F-value)
selector_kbest = SelectKBest(score_func=f_classif, k=10)
X_kbest = selector_kbest.fit_transform(X_scaled, y)

# Obtenir les noms des variables sélectionnées
selected_features_kbest = X_scaled.columns[selector_kbest.get_support()].tolist()
scores_kbest = selector_kbest.scores_

# Classement des variables
feature_scores = pd.DataFrame({
    'Feature': X_scaled.columns,
    'F-Score': scores_kbest
}).sort_values('F-Score', ascending=False)

print("Top 10 variables les plus importantes (F-Score ANOVA):")
print(feature_scores.head(10))

# Visualisation
fig, ax = plt.subplots(figsize=(12, 6))
feature_scores.head(15).sort_values('F-Score').plot(
    kind='barh', x='Feature', y='F-Score', ax=ax, color='steelblue', legend=False
)
ax.set_title('Top 15 Variables - F-Score ANOVA', fontsize=12, fontweight='bold')
ax.set_xlabel('F-Score')
plt.tight_layout()
plt.show()

print(f"\n✅ Variables sélectionnées par SelectKBest (K=10): {selected_features_kbest}")

In [ ]:
# CELL 10: Sélection de Variables - RFE avec Random Forest (Wrapper)
print("\n🎯 SÉLECTION DE VARIABLES - RFECV (Recursive Feature Elimination)")
print("="*80)

# RFE avec Random Forest comme estimateur
rf_temp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, max_depth=10)
rfe = RFE(estimator=rf_temp, n_features_to_select=10, step=1)
rfe.fit(X_scaled, y)

selected_features_rfe = X_scaled.columns[rfe.get_support()].tolist()

print(f"✅ Variables sélectionnées par RFE (n=10): {selected_features_rfe}")

# Intersection des deux méthodes (consensus)
consensus_features = list(set(selected_features_kbest) & set(selected_features_rfe))
all_selected = list(set(selected_features_kbest) | set(selected_features_rfe))

print(f"\n📊 RÉSUMÉ SÉLECTION DE VARIABLES:")
print(f"  SelectKBest (10 vars): {selected_features_kbest}")
print(f"  RFE (10 vars): {selected_features_rfe}")
print(f"  Consensus (intersection): {len(consensus_features)} variables - {consensus_features}")
print(f"  Union: {len(all_selected)} variables")

# Utiliser l'union pour plus de robustesse (12 variables)
X_selected = X_scaled[all_selected]
print(f"\n✅ Dataset final: {X_selected.shape[0]} lignes × {X_selected.shape[1]} colonnes")

## ⚖️ PHASE 4: ÉQUILIBRAGE DES DONNÉES (SMOTE)

In [ ]:
# CELL 11: Train-Test Split Stratifié
print("\n📊 TRAIN-TEST SPLIT STRATIFIÉ")
print("="*80)

# Split 70-30 avec stratification pour conserver la distribution
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, 
    test_size=0.3, 
    stratify=y, 
    random_state=42
)

print(f"Ensemble d'entraînement: {X_train.shape[0]} lignes")
print(f"Ensemble de test: {X_test.shape[0]} lignes")
print(f"\nDistribution du train set:")
print(y_train.value_counts())
print(f"\nDistribution du test set:")
print(y_test.value_counts())

In [ ]:
# CELL 12: Application de SMOTE (Oversampling)
print("\n🔄 APPLICATION DE SMOTE (OVERSAMPLING)")
print("="*80)

# SMOTE pour équilibrer les classes
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"Avant SMOTE:")
print(f"  Classe 0: {(y_train == 0).sum()}")
print(f"  Classe 1: {(y_train == 1).sum()}")
print(f"  Ratio: 1:{(y_train == 0).sum() / (y_train == 1).sum():.2f}")

print(f"\nAprès SMOTE:")
print(f"  Classe 0: {(y_train_balanced == 0).sum()}")
print(f"  Classe 1: {(y_train_balanced == 1).sum()}")
print(f"  Ratio: 1:1 (équilibré)")

print(f"\n✅ Nouveau dataset d'entraînement: {X_train_balanced.shape[0]} lignes × {X_train_balanced.shape[1]} colonnes")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Distribution du Train Set (Avant SMOTE)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Fréquence')

y_train_balanced.value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[1].set_title('Distribution du Train Set (Après SMOTE)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Fréquence')
plt.tight_layout()
plt.show()

## 🏗️ PHASE 5: MODÉLISATION - ANALYSE FACTORIELLE DISCRIMINANTE (AFD)

In [ ]:
# CELL 13: AFD - Entrainement Initial
print("\n📊 MODÈLE 1: ANALYSE FACTORIELLE DISCRIMINANTE (AFD / LDA)")
print("="*80)

# Entrainement AFD
afd_model = LinearDiscriminantAnalysis(
    solver='svd',  # SVD robuste
    n_components=1,  # Classification binaire
    random_state=42
)

afd_model.fit(X_train_balanced, y_train_balanced)

# Prédictions
y_train_pred_afd = afd_model.predict(X_train_balanced)
y_test_pred_afd = afd_model.predict(X_test)
y_train_proba_afd = afd_model.predict_proba(X_train_balanced)[:, 1]
y_test_proba_afd = afd_model.predict_proba(X_test)[:, 1]

# Métriques train/test
train_acc_afd = accuracy_score(y_train_balanced, y_train_pred_afd)
test_acc_afd = accuracy_score(y_test, y_test_pred_afd)

print(f"✅ AFD Entraîné")
print(f"\nMétriques de base:")
print(f"  Accuracy Train: {train_acc_afd:.4f}")
print(f"  Accuracy Test:  {test_acc_afd:.4f}")
print(f"  AUC-ROC Test:   {roc_auc_score(y_test, y_test_proba_afd):.4f}")

print(f"\nCoefficients (Loadings):")
print(X_selected.columns)
print(afd_model.coef_[0])

## 🏗️ PHASE 5b: MODÉLISATION - RÉGRESSION LOGISTIQUE

In [ ]:
# CELL 14: RÉGRESSION LOGISTIQUE - Entrainement Initial
print("\n📊 MODÈLE 2: RÉGRESSION LOGISTIQUE")
print("="*80)

# Régression Logistique avec class_weight pour gérer l'imbalance
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    class_weight='balanced'
)

lr_model.fit(X_train_balanced, y_train_balanced)

# Prédictions
y_train_pred_lr = lr_model.predict(X_train_balanced)
y_test_pred_lr = lr_model.predict(X_test)
y_train_proba_lr = lr_model.predict_proba(X_train_balanced)[:, 1]
y_test_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Métriques
train_acc_lr = accuracy_score(y_train_balanced, y_train_pred_lr)
test_acc_lr = accuracy_score(y_test, y_test_pred_lr)

print(f"✅ Régression Logistique Entraînée")
print(f"\nMétriques de base:")
print(f"  Accuracy Train: {train_acc_lr:.4f}")
print(f"  Accuracy Test:  {test_acc_lr:.4f}")
print(f"  AUC-ROC Test:   {roc_auc_score(y_test, y_test_proba_lr):.4f}")

# Calcul des Odds Ratios
print(f"\n📊 ODDS RATIOS (Exp des coefficients):")
odds_ratios = pd.DataFrame({
    'Variable': X_selected.columns,
    'Coefficient': lr_model.coef_[0],
    'Odds_Ratio': np.exp(lr_model.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print(odds_ratios)

# Interprétation
print(f"\n💡 INTERPRÉTATION:")
print(f"Si Odds_Ratio > 1: augmentation d'une unité augmente les chances de défaut")
print(f"Si Odds_Ratio < 1: augmentation d'une unité diminue les chances de défaut")

## 🏗️ PHASE 5c: MODÉLISATION - ARBRE DE DÉCISION

In [ ]:
# CELL 15: ARBRE DE DÉCISION - Entrainement Initial
print("\n🌳 MODÈLE 3: ARBRE DE DÉCISION")
print("="*80)

# Arbre de Décision
dt_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    class_weight='balanced'
)

dt_model.fit(X_train_balanced, y_train_balanced)

# Prédictions
y_train_pred_dt = dt_model.predict(X_train_balanced)
y_test_pred_dt = dt_model.predict(X_test)
y_train_proba_dt = dt_model.predict_proba(X_train_balanced)[:, 1]
y_test_proba_dt = dt_model.predict_proba(X_test)[:, 1]

# Métriques
train_acc_dt = accuracy_score(y_train_balanced, y_train_pred_dt)
test_acc_dt = accuracy_score(y_test, y_test_pred_dt)

print(f"✅ Arbre de Décision Entraîné")
print(f"\nMétriques de base:")
print(f"  Accuracy Train: {train_acc_dt:.4f}")
print(f"  Accuracy Test:  {test_acc_dt:.4f}")
print(f"  AUC-ROC Test:   {roc_auc_score(y_test, y_test_proba_dt):.4f}")

print(f"\n📊 COMPLEXITÉ DE L'ARBRE:")
print(f"  Profondeur réelle: {dt_model.get_depth()}")
print(f"  Nombre de feuilles: {dt_model.get_n_leaves()}")
print(f"  Nombre de nœuds: {dt_model.tree_.node_count}")

## 🏗️ PHASE 5d: MODÉLISATION - RANDOM FOREST

In [ ]:
# CELL 16: RANDOM FOREST - Entrainement Initial
print("\n🌲 MODÈLE 4: RANDOM FOREST")
print("="*80)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    oob_score=True,  # Out-Of-Bag score
    class_weight='balanced'
)

rf_model.fit(X_train_balanced, y_train_balanced)

# Prédictions
y_train_pred_rf = rf_model.predict(X_train_balanced)
y_test_pred_rf = rf_model.predict(X_test)
y_train_proba_rf = rf_model.predict_proba(X_train_balanced)[:, 1]
y_test_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métriques
train_acc_rf = accuracy_score(y_train_balanced, y_train_pred_rf)
test_acc_rf = accuracy_score(y_test, y_test_pred_rf)
oob_score_rf = rf_model.oob_score_

print(f"✅ Random Forest Entraîné")
print(f"\nMétriques de base:")
print(f"  Accuracy Train: {train_acc_rf:.4f}")
print(f"  Accuracy Test:  {test_acc_rf:.4f}")
print(f"  Out-Of-Bag Score: {oob_score_rf:.4f}")
print(f"  AUC-ROC Test:   {roc_auc_score(y_test, y_test_proba_rf):.4f}")

print(f"\n📊 CONFIGURATION:")
print(f"  Nombre d'arbres: {rf_model.n_estimators}")
print(f"  Profondeur max: {rf_model.max_depth}")
print(f"  Samples min (split): {rf_model.min_samples_split}")
print(f"  Samples min (leaf): {rf_model.min_samples_leaf}")

## 🔧 PHASE 6: FINE-TUNING DES HYPERPARAMÈTRES

In [ ]:
# CELL 17: FINE-TUNING - RÉGRESSION LOGISTIQUE (GridSearchCV)
print("\n🔍 FINE-TUNING 1: RÉGRESSION LOGISTIQUE")
print("="*80)

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'penalty': ['l2']
}

grid_search_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    param_grid_lr,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_lr.fit(X_train_balanced, y_train_balanced)

print(f"✅ Meilleurs hyperparamètres trouvés:")
print(f"  C: {grid_search_lr.best_params_['C']}")
print(f"  Solver: {grid_search_lr.best_params_['solver']}")
print(f"  Best CV Score (ROC-AUC): {grid_search_lr.best_score_:.4f}")

# Meilleur modèle
lr_best = grid_search_lr.best_estimator_
y_test_proba_lr_best = lr_best.predict_proba(X_test)[:, 1]

print(f"\n  Test AUC-ROC: {roc_auc_score(y_test, y_test_proba_lr_best):.4f}")

In [ ]:
# CELL 18: FINE-TUNING - ARBRE DE DÉCISION (GridSearchCV)
print("\n🔍 FINE-TUNING 2: ARBRE DE DÉCISION")
print("="*80)

param_grid_dt = {
    'max_depth': [5, 8, 10, 12, 15],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10, 15],
    'criterion': ['gini', 'entropy']
}

grid_search_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    param_grid_dt,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_dt.fit(X_train_balanced, y_train_balanced)

print(f"✅ Meilleurs hyperparamètres trouvés:")
for param, value in grid_search_dt.best_params_.items():
    print(f"  {param}: {value}")
print(f"  Best CV Score (ROC-AUC): {grid_search_dt.best_score_:.4f}")

# Meilleur modèle
dt_best = grid_search_dt.best_estimator_
y_test_proba_dt_best = dt_best.predict_proba(X_test)[:, 1]

print(f"\n  Test AUC-ROC: {roc_auc_score(y_test, y_test_proba_dt_best):.4f}")

In [ ]:
# CELL 19: FINE-TUNING - RANDOM FOREST (GridSearchCV)
print("\n🔍 FINE-TUNING 3: RANDOM FOREST")
print("="*80)

param_grid_rf = {
    'n_estimators': [50, 100, 150],
    'max_depth': [8, 10, 12],
    'min_samples_split': [10, 15, 20],
    'min_samples_leaf': [5, 10],
    'max_features': ['sqrt', 'log2']
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced', oob_score=True),
    param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_rf.fit(X_train_balanced, y_train_balanced)

print(f"✅ Meilleurs hyperparamètres trouvés:")
for param, value in grid_search_rf.best_params_.items():
    print(f"  {param}: {value}")
print(f"  Best CV Score (ROC-AUC): {grid_search_rf.best_score_:.4f}")

# Meilleur modèle
rf_best = grid_search_rf.best_estimator_
y_test_proba_rf_best = rf_best.predict_proba(X_test)[:, 1]
oob_score_rf_best = rf_best.oob_score_

print(f"\n  OOB Score: {oob_score_rf_best:.4f}")
print(f"  Test AUC-ROC: {roc_auc_score(y_test, y_test_proba_rf_best):.4f}")

## 📊 PHASE 7: ÉVALUATION ET COMPARAISON DES MODÈLES

In [ ]:
# CELL 20: Calcul des Métriques Détaillées pour Tous les Modèles
print("\n📊 ÉVALUATION DÉTAILLÉE DE TOUS LES MODÈLES")
print("="*100)

# Dictionnaire pour stocker les résultats
models = {
    'AFD': y_test_proba_afd,
    'Régression Logistique': y_test_proba_lr_best,
    'Arbre de Décision': y_test_proba_dt_best,
    'Random Forest': y_test_proba_rf_best
}

results = {}

for model_name, y_pred_proba in models.items():
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # Métriques
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)
    
    # Spécificité et sensibilité
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    results[model_name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall (Sensibilité)': rec,
        'Specificity': specificity,
        'F1-Score': f1,
        'AUC-ROC': auc_roc,
        'AP (Avg Precision)': ap,
        'TP': tp,
        'TN': tn,
        'FP': fp,
        'FN': fn
    }
    
    print(f"\n🎯 {model_name.upper()}")
    print("-" * 100)
    print(f"  Accuracy:      {acc:.4f}")
    print(f"  Precision:     {prec:.4f}  (TP: {tp}, FP: {fp})")
    print(f"  Recall:        {rec:.4f}  (TP: {tp}, FN: {fn})")
    print(f"  Specificity:   {specificity:.4f}  (TN: {tn})")
    print(f"  F1-Score:      {f1:.4f}")
    print(f"  AUC-ROC:       {auc_roc:.4f}")
    print(f"  Avg Precision: {ap:.4f}")

# Tableau comparatif
results_df = pd.DataFrame(results).T
print(f"\n\n📊 TABLEAU COMPARATIF")
print("="*100)
print(results_df[['Accuracy', 'Precision', 'Recall (Sensibilité)', 'Specificity', 'F1-Score', 'AUC-ROC', 'AP (Avg Precision)']].round(4))

In [ ]:
# CELL 21: Courbes ROC et AUC Comparative
print("\n📈 COURBES ROC - COMPARAISON DES MODÈLES")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, (model_name, y_pred_proba) in enumerate(models.items()):
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    axes[idx].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
    axes[idx].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    axes[idx].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
    axes[idx].set_xlim([0.0, 1.0])
    axes[idx].set_ylim([0.0, 1.05])
    axes[idx].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{model_name} - ROC Curve', fontsize=12, fontweight='bold')
    axes[idx].legend(loc="lower right", fontsize=10)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 22: Courbes Précision-Rappel (PR-AUC)
print("\n📈 COURBES PRÉCISION-RAPPEL")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, (model_name, y_pred_proba) in enumerate(models.items()):
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)
    
    axes[idx].plot(recall, precision, color='green', lw=2.5, label=f'PR curve (AP = {pr_auc:.4f})')
    axes[idx].axhline(y=y_test.sum()/len(y_test), color='red', linestyle='--', lw=2, label='Baseline (Random)')
    axes[idx].fill_between(recall, precision, alpha=0.2, color='green')
    axes[idx].set_xlim([0.0, 1.0])
    axes[idx].set_ylim([0.0, 1.05])
    axes[idx].set_xlabel('Recall', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Precision', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{model_name} - Precision-Recall Curve', fontsize=12, fontweight='bold')
    axes[idx].legend(loc="upper right", fontsize=10)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 23: Matrices de Confusion
print("\n📊 MATRICES DE CONFUSION")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for idx, (model_name, y_pred_proba) in enumerate(models.items()):
    y_pred = (y_pred_proba >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], 
                cbar_kws={'label': 'Count'}, annot_kws={'size': 14, 'weight': 'bold'})
    
    axes[idx].set_xlabel('Predicted', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Actual', fontsize=11, fontweight='bold')
    axes[idx].set_title(f'{model_name} - Confusion Matrix', fontsize=12, fontweight='bold')
    axes[idx].set_xticklabels(['No Default', 'Default'])
    axes[idx].set_yticklabels(['No Default', 'Default'])

plt.tight_layout()
plt.show()

## 🎯 PHASE 8: FEATURE IMPORTANCE ANALYSIS

In [ ]:
# CELL 24: Feature Importance - Arbre de Décision et Random Forest
print("\n🎯 FEATURE IMPORTANCE - ARBRE DE DÉCISION")
print("="*80)

dt_importances = dt_best.feature_importances_
dt_features_importance = pd.DataFrame({
    'Feature': X_selected.columns,
    'Importance': dt_importances
}).sort_values('Importance', ascending=False)

print("Top 10 variables importantes (Arbre de Décision):")
print(dt_features_importance.head(10))

print("\n🎯 FEATURE IMPORTANCE - RANDOM FOREST")
print("="*80)

rf_importances = rf_best.feature_importances_
rf_features_importance = pd.DataFrame({
    'Feature': X_selected.columns,
    'Importance': rf_importances
}).sort_values('Importance', ascending=False)

print("Top 10 variables importantes (Random Forest):")
print(rf_features_importance.head(10))

# Visualisation comparée
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

dt_features_importance.head(10).sort_values('Importance').plot(
    kind='barh', x='Feature', y='Importance', ax=axes[0], color='steelblue', legend=False
)
axes[0].set_title('Top 10 Features - Decision Tree', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')

rf_features_importance.head(10).sort_values('Importance').plot(
    kind='barh', x='Feature', y='Importance', ax=axes[1], color='forestgreen', legend=False
)
axes[1].set_title('Top 10 Features - Random Forest', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 25: Coefficients Détaillés - Régression Logistique Optimisée
print("\n📊 ANALYSE DÉTAILLÉE - RÉGRESSION LOGISTIQUE OPTIMISÉE")
print("="*100)

# Odds Ratios avec intervalles de confiance (approximation de Wald)
from scipy import stats as sp_stats

coeffs = lr_best.coef_[0]
odds_ratios_detailed = pd.DataFrame({
    'Variable': X_selected.columns,
    'Coefficient': coeffs,
    'Odds_Ratio': np.exp(coeffs),
    'Exp(Coeff)': np.exp(coeffs)
}).sort_values('Odds_Ratio', ascending=False)

odds_ratios_detailed['% Change'] = (odds_ratios_detailed['Odds_Ratio'] - 1) * 100

print("ODDS RATIOS DÉTAILLÉS:")
print(odds_ratios_detailed.to_string())

print("\n\n💡 INTERPRÉTATION STATISTIQUE:")
print("-" * 100)
for idx, row in odds_ratios_detailed.head(5).iterrows():
    if row['Odds_Ratio'] > 1:
        pct = row['% Change']
        print(f"  • {row['Variable']}: OR = {row['Odds_Ratio']:.4f}")
        print(f"    → Augmentation de 1 unité = {pct:.2f}% d'augmentation des chances de défaut")
    else:
        pct = abs(row['% Change'])
        print(f"  • {row['Variable']}: OR = {row['Odds_Ratio']:.4f}")
        print(f"    → Augmentation de 1 unité = {pct:.2f}% de diminution des chances de défaut")

## 📋 PHASE 9: RÉSUMÉ FINAL ET RECOMMANDATIONS

In [ ]:
# CELL 26: Résumé Comparatif Final
print("\n" + "="*100)
print("🏆 RÉSUMÉ COMPARATIF FINAL - CLASSEMENT DES MODÈLES")
print("="*100 + "\n")

# Créer un tableau de synthèse
synthesis = results_df[['Accuracy', 'Precision', 'Recall (Sensibilité)', 'F1-Score', 'AUC-ROC', 'AP (Avg Precision)']].round(4)

# Ajouter le classement
for metric in synthesis.columns:
    synthesis[f'{metric}_Rank'] = synthesis[metric].rank(ascending=False)

print("📊 TABLEAU SYNTHÉTIQUE (Métriques principales):")
print(synthesis[['Accuracy', 'Precision', 'Recall (Sensibilité)', 'F1-Score', 'AUC-ROC', 'AP (Avg Precision)']].to_string())

print("\n\n🏆 MEILLEUR MODÈLE PAR MÉTRIQUE:")
print("-" * 100)
for metric in ['Accuracy', 'Precision', 'Recall (Sensibilité)', 'F1-Score', 'AUC-ROC', 'AP (Avg Precision)']:
    best_model = results_df[metric].idxmax()
    best_value = results_df[metric].max()
    print(f"  {metric:30s}: {best_model:30s} ({best_value:.4f})")

print("\n\n💡 ANALYSE GLOBALE:")
print("-" * 100)

# Score composite (moyenne pondérée)
weights = {
    'Accuracy': 0.15,
    'Precision': 0.15,
    'Recall (Sensibilité)': 0.20,
    'F1-Score': 0.20,
    'AUC-ROC': 0.20,
    'AP (Avg Precision)': 0.10
}

composite_scores = {}
for model in results_df.index:
    score = sum(results_df.loc[model, metric] * weights[metric] for metric in weights)
    composite_scores[model] = score

best_overall = max(composite_scores, key=composite_scores.get)
print(f"\n✅ MEILLEUR MODÈLE GLOBAL: {best_overall}")
print(f"\nScore composite (moyenne pondérée):")
for model, score in sorted(composite_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model:30s}: {score:.4f}")

In [ ]:
# CELL 27: Visualisation Comparée des Modèles
print("\n📊 VISUALISATION COMPARATIVE")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Comparaison des AUC-ROC
metrics_to_plot = ['Accuracy', 'AUC-ROC', 'F1-Score', 'Recall (Sensibilité)']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    values = results_df[metric].sort_values(ascending=False)
    colors = ['#27ae60' if v == values.max() else '#3498db' for v in values]
    
    bars = ax.bar(range(len(values)), values, color=colors, edgecolor='black', linewidth=1.5)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(values.index, rotation=45, ha='right')
    ax.set_ylabel(metric, fontweight='bold', fontsize=11)
    ax.set_title(f'Comparaison: {metric}', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', 
                ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 28: Recommandations Finales
print("\n" + "="*100)
print("🎯 RECOMMANDATIONS FINALES")
print("="*100 + "\n")

print("1️⃣  CHOIX DU MODÈLE:")
print("-" * 100)
print(f"""
   ✅ MEILLEUR MODÈLE GLOBAL: {best_overall}
   
   Justifications:
   • Meilleur équilibre entre les métriques importantes
   • Score composite pondéré: {composite_scores[best_overall]:.4f}
   • ROC-AUC: {results_df.loc[best_overall, 'AUC-ROC']:.4f}
   • F1-Score: {results_df.loc[best_overall, 'F1-Score']:.4f}
""")

print("\n2️⃣  INTERPRÉTATION MÉTHODOLOGIQUE:")
print("-" * 100)
print("""
   📊 ÉTAPES MÉTHODOLOGIQUES RESPECTÉES:
   ✓ Exploration et nettoyage des données
   ✓ Gestion des valeurs manquantes (imputation médiane/mode)
   ✓ Traitement des outliers (Winsorization)
   ✓ Création de variables (Feature Engineering)
   ✓ Sélection de variables (SelectKBest + RFE)
   ✓ Équilibrage des données (SMOTE)
   ✓ Normalisation robuste des variables
   ✓ Split stratifié Train/Test (70/30)
   ✓ Modélisation de 4 algorithmes
   ✓ Fine-tuning des hyperparamètres (GridSearchCV)
   ✓ Évaluation multimétrique (ROC/AUC/PR/F1)
   ✓ Analyse de Feature Importance
""")

print("\n3️⃣  PERFORMANCES COMPARÉES:")
print("-" * 100)
for model in sorted(results_df.index):
    score = composite_scores[model]
    print(f"   {model:30s} - Score: {score:.4f} - AUC-ROC: {results_df.loc[model, 'AUC-ROC']:.4f}")

print("\n4️⃣  RECOMMANDATIONS PRATIQUES:")
print("-" * 100)
print("""
   Pour un système de prédiction de défaut de crédit:
   
   • Préférer la Sensibilité (Recall) à la Précision pour minimiser les faux négatifs
     (mieux détecter tous les défauts possibles)
   
   • Utiliser un seuil de probabilité < 0.5 pour augmenter la détection
   
   • Variables les plus importantes à monitorer:
     1. Late payment score (paiements tardifs pondérés)
     2. Age et dérivés
     3. Debt Ratio
     4. Monthly Income
     5. Credit utilization risk
   
   • Réentraîner le modèle régulièrement avec nouvelles données
     (drift concept possible dans le temps)
""")